# Live lab 4 · A completed statistical workflow with a real LLM interpreter

Read after Chapter 14. The LLM translates the natural-language request. A tested tool computes coverage. An explicitly deterministic template reports the tool fields. This is a declared workflow, not a hidden replacement for a failed LLM-written report.

**Live execution.** Restarting and running this notebook performs new local model inference. Saved outputs are one actual development run, not guaranteed answers. No paid API is used. First follow [Python setup](https://chikuang.github.io/teaching/genai/python-workflow.html), then run these commands in a terminal at the cloned course root:

```bash
python -m pip install -r requirements.txt -r requirements-llm.txt
python labs/download_model.py
python -m jupyterlab
```

Open the notebook in JupyterLab. Press **Shift+Enter** in order. Model weights take about 3 GB on disk; CPU float32 inference needs several additional GB of RAM. The first model-loading cell can take time. Do not run it repeatedly while it is busy. Inference remains on your machine.


In [ ]:
from pathlib import Path
import os
import sys

start = Path(os.environ.get('GENAI_COURSE_ROOT', Path.cwd())).resolve()
root = None
# Examine this folder, then each enclosing folder.
candidates = [start] + list(start.parents)
for candidate in candidates:
    if (candidate / 'labs/model.json').is_file():
        root = candidate
        break
if root is None:
    raise FileNotFoundError('Keep the notebook inside Teaching-GenAI, or set GENAI_COURSE_ROOT')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import json
from labs.llm_lab import LocalLM
from labs.notebook_support import save_run
print('Course folder:', root.name)
print('Python:', sys.version.split()[0])

In [ ]:
lm = LocalLM(root / '_models/qwen2.5-1.5b')
print(lm.versions)
print('Model loaded; no question has been generated yet.')

## 1. Declare full request cards before inference

A request card has six fields: sampling family, sample size, interval method, nominal confidence, simulation repetitions, and dependence. It represents a person's review of the original question. It contains no estimated coverage or numerical answer. These cards are deliberately supplied for this teaching demonstration; they are not an automatic natural-language verifier.

The live workflow supports **iid normal/exponential, n=5–500, nominal 95%, z with estimated SD, 500 repetitions**. Every other setting is refused explicitly. The numerical tool can support larger budgets, but this live protocol fixes its budget in advance. Without a reviewed card, a model proposal is returned as `needs_confirmation` and no simulation runs.

The seven authored requests include changes in each unsupported dimension, so normal family and correct n alone cannot pass. They are development cases, not a held-out population sample.

In [ ]:
from labs.guided_systems import interpret_scenario, execute_confirmed, simulation_workflow, scope_reasons
from labs.live_tasks import CAPSTONE_CASES
from scipy.stats import t
import numpy as np

for task_id, question, card in CAPSTONE_CASES:
    print(task_id, card)
    print('Scope restrictions:', scope_reasons(card))

## 2. Expand the first request, then use the wrapper

`interpret_scenario` returns a Python dictionary after a real LLM call and structural validation. Its raw answer remains in `lm.calls`, including rejected outputs. A dictionary with all the correct types can still encode the wrong intent.

`execute_confirmed` first requires a separately reviewed card, checks the declared scope, and compares **all six fields**. Only an exact supported match calls the simulator. A mismatch is an experimental failure; no different task is silently computed. The tool returns numerical evidence, and a deterministic template renders it. This protects execution conditional on the reviewed card being correct; it cannot prove that a human captured every ambiguity in free text.

The first iteration below expands proposal and execution as two lines. Remaining iterations call the wrapper that performs the same operations. Each task makes one actual interpreter call. `call_ids` are zero-based indices linking rows to exact frozen prompts and raw outputs.

In [ ]:
rows = []
for index, (task_id, question, card) in enumerate(CAPSTONE_CASES):
    start_call = len(lm.calls)
    if index == 0:
        row = {'question': question, 'confirmed_request': card,
               'report_kind': 'deterministic_template'}
        try:
            proposal = interpret_scenario(lm, question)
            print('Proposed dictionary:', proposal)
            row['request'] = proposal
            row.update(execute_confirmed(proposal, card))
        except (ValueError, TypeError) as error:
            row.update(status='invalid', error=str(error))
    else:
        row = simulation_workflow(lm, question, confirmed_request=card)
    row['task_id'] = task_id
    row['call_ids'] = list(range(start_call, len(lm.calls)))
    row['intent_correct'] = row.get('request') == card
    expected_status = 'abstained' if scope_reasons(card) else 'completed'
    row['task_success'] = row['intent_correct'] and row['status'] == expected_status
    rows.append(row)
    print(task_id, row['status'], 'intent correct:', row['intent_correct'])
    print(row.get('report', row.get('error')))
    if scope_reasons(card):
        assert 'tool_result' not in row
assert len(lm.calls) == len(CAPSTONE_CASES)
# Even a valid proposal cannot execute before independent review.
assert execute_confirmed(CAPSTONE_CASES[0][2], None)['status'] == 'needs_confirmation'
wrong_proposal = dict(CAPSTONE_CASES[0][2], family='exponential')
assert execute_confirmed(wrong_proposal, CAPSTONE_CASES[0][2])['status'] == 'intent_mismatch' 

## 3. Check the statistical meaning independently

For normal samples the target is `2*F_t(1.96)-1`, because the procedure uses 1.96 with an estimated standard error. The Monte Carlo interval targets that coverage probability. It is different from the mean interval computed inside each simulated dataset. Five standard errors is a reproducible diagnostic tolerance, not a mathematical guarantee that random checks never fail.

In [ ]:
for row in rows:
    if row['status'] != 'completed':
        continue
    result = row['tool_result']
    low, high = result['mc_interval']
    assert 0 <= low <= high <= 1
    assert f"{result['estimate']:.3f}" in row['report']
    if result['family'] == 'normal':
        reference = 2 * t.cdf(1.96, result['n'] - 1) - 1
        mc_se = np.sqrt(reference * (1-reference) / result['repetitions'])
        print('Analytic normal coverage:', round(reference, 4))
        assert abs(result['estimate'] - reference) < 5 * mc_se
print('Descriptive task successes:', sum(r['task_success'] for r in rows), '/', len(rows))
save_run(root, 'capstone-bridge', lm, rows)

## 4. Add LLM-written reporting only as a separate experiment

The current report is deliberately a Python template. A next experiment may give the same tool JSON to an LLM. Preserve its raw report even if rejected; never silently substitute this template and mark the LLM report successful. Score exact family and sample size, numeric agreement at declared precision, distinction between coverage and confidence level, and absence of universal guarantees. See the failed real reports in Appendix B.

## Practice

1. **Predict:** which error can survive schema validation while producing a perfectly correct calculation for the wrong task?
2. **Implement:** use a scripted normal request for the exponential question. The numerical checks can pass, but `intent_correct` must be false. A Boolean sample size must be rejected before simulation.
3. **Investigate:** freeze a revised prompt before new requests, include unsupported families, and separately report interpretation, execution and narrative scores. A total success count over these three authored cases is descriptive only.

**Transfer assignment:** start from [Assignment 14](https://chikuang.github.io/teaching/genai/assignments.html). Propose one independently reviewed scenario, freeze all six requested fields, and document whether it is supported before inference. Changing the interval method requires a new tool, an analytic reference, and a new protocol version; do not relabel a 95% z result as a 99% Student-t result.